# CRNN OCR Pipeline
## GSoC 2026 — Optical Character Recognition of Historical Printed Sources

### Architecture Overview
This notebook implements a **CNN-RNN (CRNN)** pipeline for OCR of 17th century  printed sources.

**Pipeline stages:**
```
Raw Page Image
     ↓
[1] Preprocessing       — grayscale, denoise, deskew, CLAHE, resize
     ↓
[2] Text Region Detection — isolate main text, ignore marginalia
     ↓
[3] Line Segmentation   — split page into individual text lines
     ↓
[4] CRNN Model          — CNN (VGG-style) + BiLSTM + CTC decoder
     ↓
[5] LLM Post-correction — GPT-4o-mini / Gemini late-stage cleanup
     ↓
[6] Evaluation          — CER, WER per stage
```

**Why CRNN?**
- CNN extracts local visual features from each image column
- BiLSTM captures long-range sequential dependencies in text
- CTC loss allows training without character-level segmentation labels
- Well-suited for historical printed text with variable spacing and letterforms

In [35]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [36]:
# ============================================================
# CELL 1 — SETUP: Mount Drive + Create Directory Structure
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys
from pathlib import Path
BASE_PATH = Path.cwd()
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)

dirs = [
    'data/images', 'data/annotations', 'data/lines',
    'src', 'models/crnn', 'results'
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Make src a Python package
from pathlib import Path
Path(BASE_PATH, 'src', '__init__.py').touch(exist_ok=True)

sys.path.insert(0, BASE_PATH)

print(f'✓ Working directory: {os.getcwd()}')
print(f'✓ Directories ready')

Mounted at /content/drive
✓ Working directory: /content/drive/MyDrive/OCR_Pipeline_Research
✓ Directories ready


In [ ]:

# CELL 0b — DATASET: Download RODRIGO Corpus from Zenodo
# 15,010 line images of 1545 old Spanish manuscript
# Free, CC-BY 4.0, directly usable for CRNN training


import os, sys, tarfile
from pathlib import Path

BASE_PATH = "/content/drive/MyDrive/OCR_Pipeline_Research"
RODRIGO_DIR = f"{BASE_PATH}/data/rodrigo"
os.makedirs(RODRIGO_DIR, exist_ok=True)

TAR_PATH = f"{RODRIGO_DIR}/rodrigo.tar.gz"
ZENODO_URL = "https://zenodo.org/records/1490009/files/Rodrigo%20corpus%201.0.0.tar.gz?download=1"

# --- Step 1: Download ---
if not Path(TAR_PATH).exists():
    print("Downloading RODRIGO corpus (~400MB)... this takes 2-5 min")
    !wget -q --show-progress -O {TAR_PATH} {ZENODO_URL}
    print("✓ Download complete")
else:
    print("✓ Archive already downloaded")

# --- Step 2: Extract ---
EXTRACT_DIR = f"{RODRIGO_DIR}/extracted"
if not Path(EXTRACT_DIR).exists():
    print("Extracting archive...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !tar -xzf {TAR_PATH} -C {EXTRACT_DIR}
    print("✓ Extraction complete")
else:
    print("✓ Already extracted")

# --- Step 3: Inspect structure ---
print("Dataset structure:")
!find {EXTRACT_DIR} -maxdepth 3 | head -40


In [38]:

# CELL 0c — RODRIGO: Parse GT (NO file copying — use images in place)


import os
from pathlib import Path

BASE_PATH   = "/content/drive/MyDrive/OCR_Pipeline_Research"
EXTRACT_DIR = f"{BASE_PATH}/data/rodrigo/extracted"
IMAGES_DIR  = f"{EXTRACT_DIR}/images"
TRANS_FILE  = f"{EXTRACT_DIR}/text/transcriptions.txt"
TRAIN_FILE  = f"{EXTRACT_DIR}/partitions/train.txt"
VAL_FILE    = f"{EXTRACT_DIR}/partitions/validation.txt"
ANNOT_DIR   = f"{BASE_PATH}/data/annotations"
os.makedirs(ANNOT_DIR, exist_ok=True)

# Step 1: Load transcriptions
print("Loading transcriptions...")
transcriptions = {}
with open(TRANS_FILE, encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split(" ", 1) if "\t" not in line else line.split("\t", 1)
        if len(parts) == 2:
            key = parts[0].strip().replace(".png", "")
            transcriptions[key] = parts[1].strip()

print(f"  Loaded {len(transcriptions)} transcriptions")
print(f"  Sample keys: {list(transcriptions.keys())[:3]}")

# Step 2: Load partition lists
def load_partition(filepath):
    names = set()
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            name = line.strip().replace(".png", "")
            if name:
                names.add(name)
    return names

train_names = load_partition(TRAIN_FILE)
val_names   = load_partition(VAL_FILE)
print(f"  Train: {len(train_names)} | Val: {len(val_names)}")

# Step 3: Build label files pointing DIRECTLY to original image paths
# No copying needed — dataset.py will read from IMAGES_DIR
def build_labels(names, split_name, image_dir):
    labels = []
    missing = 0
    for name in sorted(names):
        if name not in transcriptions:
            missing += 1
            continue
        img_path = Path(image_dir) / f"{name}.png"
        if not img_path.exists():
            missing += 1
            continue
        # Store FULL absolute path so dataset.py finds it directly
        labels.append(f"{img_path}	{transcriptions[name]}")
    print(f"  {split_name}: {len(labels)} samples | {missing} skipped")
    return labels

print("Building label files (no copying)...")
train_labels = build_labels(train_names, "train", IMAGES_DIR)
val_labels   = build_labels(val_names,   "val",   IMAGES_DIR)

# Save label files
with open(f"{ANNOT_DIR}/line_labels.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_labels))

with open(f"{ANNOT_DIR}/line_labels_val.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(val_labels))

print(f"✓ line_labels.txt:     {len(train_labels)} samples")
print(f"✓ line_labels_val.txt: {len(val_labels)} samples")
print(f"Sample entries:")
for e in train_labels[:3]:
    print(f"  {e[:100]}")


Loading transcriptions...
  Loaded 15010 transcriptions
  Sample keys: ['Rodrigo_00006_00', 'Rodrigo_00006_01', 'Rodrigo_00006_02']
  Train: 9000 | Val: 1000
Building label files (no copying)...
  train: 9000 samples | 0 skipped
  val: 1000 samples | 0 skipped
✓ line_labels.txt:     9000 samples
✓ line_labels_val.txt: 1000 samples
Sample entries:
  /content/drive/MyDrive/OCR_Pipeline_Research/data/rodrigo/extracted/images/Rodrigo_00006_00.png	Hist
  /content/drive/MyDrive/OCR_Pipeline_Research/data/rodrigo/extracted/images/Rodrigo_00006_01.png	Arço
  /content/drive/MyDrive/OCR_Pipeline_Research/data/rodrigo/extracted/images/Rodrigo_00006_02.png	go. 


In [ ]:
# ============================================================
# CELL 0d — VERIFY: Visualize sample RODRIGO lines
# ============================================================

import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import random

BASE_PATH = "/content/drive/MyDrive/OCR_Pipeline_Research"
LINES_DIR = f"{BASE_PATH}/data/lines"
LABELS_FILE = f"{BASE_PATH}/data/annotations/line_labels.txt"

# Load labels
samples = []
with open(LABELS_FILE, encoding="utf-8") as f:
    for line in f:
        if "	" in line:
            img, txt = line.strip().split("	", 1)
            samples.append((img, txt))

print(f"Total training samples: {len(samples)}")

# Show 6 random samples
picks = random.sample(samples, min(6, len(samples)))
fig, axes = plt.subplots(len(picks), 1, figsize=(14, len(picks) * 2))
if len(picks) == 1:
    axes = [axes]

for ax, (img_path, text) in zip(axes, picks):
    img_path = img_path.strip()


    if not Path(img_path).exists():
        print("❌ File missing:", img_path)
        continue

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print("❌ OpenCV failed:", img_path)
        continue

    ax.imshow(img, cmap="gray", aspect="auto")
    ax.set_title(f"{text[:80]}", fontsize=9, pad=4)
    ax.axis("off")

plt.suptitle("RODRIGO Dataset — Sample Lines (1545 Old Spanish)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
print("✓ Dataset ready for CRNN training!")


In [40]:
!ls /content/drive/MyDrive/OCR_Pipeline_Research/data/lines | head -20

In [41]:
# ============================================================
# CELL 2 — INSTALL DEPENDENCIES
# ============================================================

print('Installing dependencies...')

!pip install -q opencv-python pillow numpy matplotlib pandas
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q editdistance tqdm pyyaml
!pip install -q pdf2image
!apt-get install -qq poppler-utils tesseract-ocr

# For LLM post-correction (optional — needs API key)
!pip install -q openai

print('\n✓ All dependencies installed!')

Installing dependencies...
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...

✓ All dependencies installed!


In [42]:
# ============================================================
# CELL 3 — PREPROCESSING MODULE
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/preprocess.py
"""
src/preprocess.py
Image preprocessing for historical OCR.
Pipeline: grayscale -> denoise -> deskew -> CLAHE -> resize
"""

import cv2
import numpy as np
from pathlib import Path
import logging

logger = logging.getLogger(__name__)


class ImagePreprocessor:

    def __init__(self, target_height=64):
        # CRNN expects fixed height, variable width
        self.target_height = target_height

    def process(self, image_path: str):
        image = cv2.imread(str(image_path))
        if image is None:
            raise FileNotFoundError(f'Cannot read: {image_path}')

        steps = {}

        # 1. Grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        steps['gray'] = gray.copy()

        # 2. Denoise
        denoised = cv2.fastNlMeansDenoising(gray, h=10,
                                             templateWindowSize=7,
                                             searchWindowSize=21)
        steps['denoised'] = denoised.copy()

        # 3. Deskew
        deskewed = self._deskew(denoised)
        steps['deskewed'] = deskewed.copy()

        # 4. CLAHE contrast enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(deskewed)
        steps['enhanced'] = enhanced.copy()

        return enhanced, steps

    def _deskew(self, image: np.ndarray) -> np.ndarray:
        _, binary = cv2.threshold(image, 0, 255,
                                   cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        coords = np.column_stack(np.where(binary > 0))
        if len(coords) < 10:
            return image
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = 90 + angle
        if abs(angle) < 0.5:
            return image
        h, w = image.shape
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        return cv2.warpAffine(image, M, (w, h),
                               flags=cv2.INTER_CUBIC,
                               borderMode=cv2.BORDER_REPLICATE)

    def resize_for_crnn(self, image: np.ndarray) -> np.ndarray:
        """Resize to fixed height (CRNN input), preserve aspect ratio."""
        h, w = image.shape[:2]
        scale = self.target_height / h
        new_w = max(1, int(w * scale))
        return cv2.resize(image, (new_w, self.target_height),
                          interpolation=cv2.INTER_CUBIC)


print('✓ preprocess.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/preprocess.py


In [43]:
# ============================================================
# CELL 4 — TEXT REGION DETECTION (ignore marginalia)
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/region_detector.py
"""
src/region_detector.py
Detects the main text block in a page, ignoring marginalia.

Strategy:
- Project pixel density onto X and Y axes
- Find the largest contiguous dense region (main text block)
- Crop everything outside that region
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt


class TextRegionDetector:

    def __init__(self, margin_ratio=0.15):
        """
        margin_ratio: fraction of page width to ignore on left/right
                      as potential marginalia zone
        """
        self.margin_ratio = margin_ratio

    def detect(self, image: np.ndarray, visualize=False):
        """
        Detect main text region.
        Returns cropped image containing only main text block.
        """
        h, w = image.shape[:2]

        # Binary threshold
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        _, binary = cv2.threshold(gray, 0, 255,
                                   cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Morphological closing to connect text regions
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 5))
        closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

        # Find contours
        contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            return image, (0, 0, w, h)

        # Filter: keep only contours in the central region (ignore margins)
        margin = int(w * self.margin_ratio)
        central_contours = []
        for c in contours:
            x, y, cw, ch = cv2.boundingRect(c)
            # Must overlap with central area
            if x + cw > margin and x < w - margin:
                # Must be large enough to be text (not noise)
                if cw * ch > (w * h * 0.001):
                    central_contours.append(c)

        if not central_contours:
            return image, (0, 0, w, h)

        # Bounding box around all central contours = main text block
        all_pts = np.concatenate(central_contours)
        x, y, bw, bh = cv2.boundingRect(all_pts)

        # Add small padding
        pad = 10
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(w, x + bw + pad)
        y2 = min(h, y + bh + pad)

        cropped = image[y1:y2, x1:x2]

        if visualize:
            vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if len(image.shape) == 2 else image.copy()
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            plt.title('Detected Main Text Region (green box)')
            plt.axis('off')
            plt.show()

        return cropped, (x1, y1, x2, y2)


print('✓ region_detector.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/region_detector.py


In [44]:
# ============================================================
# CELL 5 — LINE SEGMENTATION
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/line_segmenter.py
"""
src/line_segmenter.py
Splits a text block image into individual text lines.
Uses horizontal projection profile (pixel density per row).
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import List


class LineSegmenter:

    def __init__(self, min_line_height=15, gap_threshold=5):
        self.min_line_height = min_line_height
        self.gap_threshold = gap_threshold

    def segment(self, image: np.ndarray, visualize=False) -> List[np.ndarray]:
        """
        Segment image into text line strips.
        Returns list of line images.
        """
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        # Binarize
        _, binary = cv2.threshold(gray, 0, 255,
                                   cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Horizontal projection: sum of dark pixels per row
        h_proj = np.sum(binary, axis=1)

        # Smooth the projection
        h_proj_smooth = np.convolve(h_proj, np.ones(3)/3, mode='same')

        # Find text rows (where projection > threshold)
        threshold = np.max(h_proj_smooth) * 0.05
        is_text_row = h_proj_smooth > threshold

        # Find line boundaries
        lines = []
        in_line = False
        start = 0
        gap_count = 0

        for row_idx, is_text in enumerate(is_text_row):
            if is_text:
                if not in_line:
                    start = row_idx
                    in_line = True
                gap_count = 0
            else:
                if in_line:
                    gap_count += 1
                    if gap_count >= self.gap_threshold:
                        end = row_idx - gap_count
                        if end - start >= self.min_line_height:
                            lines.append((start, end))
                        in_line = False
                        gap_count = 0

        # Don't forget last line
        if in_line:
            end = len(is_text_row)
            if end - start >= self.min_line_height:
                lines.append((start, end))

        # Crop line images with small vertical padding
        pad = 3
        h, w = image.shape[:2]
        line_images = []
        for (y1, y2) in lines:
            y1p = max(0, y1 - pad)
            y2p = min(h, y2 + pad)
            line_img = gray[y1p:y2p, :]
            line_images.append(line_img)

        if visualize and lines:
            vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
            for (y1, y2) in lines:
                cv2.rectangle(vis, (0, y1), (w, y2), (0, 200, 0), 1)
            plt.figure(figsize=(14, 10))
            plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            plt.title(f'Line Segmentation: {len(lines)} lines detected')
            plt.axis('off')
            plt.show()

        return line_images, lines


print('✓ line_segmenter.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/line_segmenter.py


In [45]:
%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/crnn_model.py
"""
src/crnn_model.py
CRNN: Convolutional Recurrent Neural Network for OCR

Architecture:
  Input image (1 x H x W)  — grayscale, fixed height=64
       ↓
  CNN Feature Extractor     — VGG-style: Conv→BN→ReLU→Pool (x5)
       ↓
  Feature Map (C x 1 x T)   — T = W/16 time steps
       ↓
  Reshape → Sequence (T x C)
       ↓
  BiLSTM (2 layers)          — captures left+right context
       ↓
  Linear → logits (T x vocab)
       ↓
  CTC Decoder                — produces final character sequence
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


class ConvBlock(nn.Module):
    """Conv2d + BatchNorm + ReLU block"""

    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class CNN(nn.Module):
    """
    VGG-style CNN feature extractor.
    Input:  (B, 1, 64, W)
    Output: (B, 512, 1, W//16)
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1 — (B,1,64,W) -> (B,64,32,W/2)
            ConvBlock(1, 64),
            nn.MaxPool2d(2, 2),

            # Block 2 — (B,64,32,W/2) -> (B,128,16,W/4)
            ConvBlock(64, 128),
            nn.MaxPool2d(2, 2),

            # Block 3 — (B,128,16,W/4) -> (B,256,8,W/4)
            ConvBlock(128, 256),
            ConvBlock(256, 256),
            nn.MaxPool2d((2, 1), (2, 1)),   # pool only height

            # Block 4 — (B,256,8,W/4) -> (B,512,4,W/4)
            ConvBlock(256, 512),
            ConvBlock(512, 512),
            nn.MaxPool2d((2, 1), (2, 1)),   # pool only height

            # Block 5 — (B,512,4,W/4) -> (B,512,1,W/4)
            ConvBlock(512, 512, kernel=(4, 1), padding=0),
        )

    def forward(self, x):
        return self.features(x)


class CRNN(nn.Module):
    """
    Full CRNN model.

    Args:
        vocab_size: number of unique characters + 1 (CTC blank)
        hidden_size: BiLSTM hidden units (per direction)
        num_rnn_layers: number of stacked BiLSTM layers
    """

    def __init__(self, vocab_size: int, hidden_size: int = 256,
                 num_rnn_layers: int = 2, dropout: float = 0.3):
        super().__init__()

        self.cnn = CNN()
        cnn_out_channels = 512

        self.rnn = nn.LSTM(
            input_size=cnn_out_channels,
            hidden_size=hidden_size,
            num_layers=num_rnn_layers,
            bidirectional=True,
            dropout=dropout if num_rnn_layers > 1 else 0,
            batch_first=False        # (T, B, C) convention
        )

        # Output: BiLSTM gives hidden_size*2
        self.classifier = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, 1, 64, W)
        returns: (T, B, vocab_size) log-softmax output for CTC
        """
        # CNN features: (B, 512, 1, T)
        features = self.cnn(x)

        # Squeeze height dim: (B, 512, T)
        features = features.squeeze(2)

        # Permute to (T, B, 512) for LSTM
        features = features.permute(2, 0, 1)

        # BiLSTM: (T, B, hidden*2)
        rnn_out, _ = self.rnn(features)

        # Classifier: (T, B, vocab_size)
        logits = self.classifier(rnn_out)

        # Log-softmax for CTC loss
        return F.log_softmax(logits, dim=2)


print('✓ crnn_model.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/crnn_model.py


In [46]:
# ============================================================
# CELL 7 — CHARSET + DATASET
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/charset.py
"""
src/charset.py
Character set for 17th century Spanish printed sources.
Includes standard Latin chars + Spanish diacritics + historical symbols.
"""

# Spanish Renaissance character set
CHARS = (
    ' !\"#&\'()*+,-./:;?'
    'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    'abcdefghijklmnopqrstuvwxyz'
    '0123456789'
    'ÁÉÍÓÚÜÑáéíóúüñ'
    'ÀÈÌÒÙàèìòù'
    'ÂÊÎÔÛâêîôû'
    'çÇ'
    # Common in early modern Spanish print
    'ÿœæ£§†‡'
)

# CTC blank token is index 0
BLANK_IDX = 0

# Build char -> index and index -> char mappings
# Index 0 reserved for CTC blank
char_to_idx = {c: i + 1 for i, c in enumerate(CHARS)}
idx_to_char = {i + 1: c for i, c in enumerate(CHARS)}
idx_to_char[BLANK_IDX] = ''   # blank decodes to empty

VOCAB_SIZE = len(CHARS) + 1   # +1 for blank


def encode(text: str):
    """Encode text string to list of indices. Unknown chars skipped."""
    return [char_to_idx[c] for c in text if c in char_to_idx]


def decode(indices):
    """Decode list of indices to string, collapsing CTC repeats."""
    result = []
    prev = None
    for idx in indices:
        if idx != BLANK_IDX and idx != prev:
            result.append(idx_to_char.get(idx, ''))
        prev = idx
    return ''.join(result)


print(f'✓ charset.py saved — vocab size: {VOCAB_SIZE}')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/charset.py


In [47]:
# ============================================================
# CELL 8 — DATASET CLASS
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/dataset.py
"""
src/dataset.py
PyTorch Dataset for CRNN training.
Expects: data/lines/<image_name>.png + data/annotations/labels.txt

labels.txt format (tab-separated):
    line_001.png\tEl rey don Alfonso
    line_002.png\tque era muy pequeño
"""

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path
from src.charset import encode, VOCAB_SIZE
import sys
sys.path.insert(0, '/content/drive/MyDrive/OCR_Pipeline_Research')


class OCRDataset(Dataset):

    def __init__(self, labels_file: str, image_dir: str,
                 target_height=64, target_width=512, augment=False):
        self.image_dir = Path(image_dir)
        self.target_height = target_height
        self.target_width = target_width
        self.augment = augment
        self.samples = []

        with open(labels_file, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split('\t', 1)
                if len(parts) == 2:
                    self.samples.append((parts[0].strip(), parts[1].strip()))

        print(f'Dataset loaded: {len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, text = self.samples[idx]
        if filename.startswith("/") or filename.startswith("content"):
            img_path = Path("/" + filename.lstrip("/"))  # normalize path
        else:
            img_path = self.image_dir / filename
        print("Loading:", img_path)

        image = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            # Return blank image if file missing
            image = np.ones((self.target_height, self.target_width),
                            dtype=np.uint8) * 255

        image = self._resize(image)

        if self.augment:
            image = self._augment(image)

        # Normalize to [0, 1], add channel dim
        image = image.astype(np.float32) / 255.0
        image = torch.tensor(image).unsqueeze(0)   # (1, H, W)

        label = encode(text)
        label_tensor = torch.tensor(label, dtype=torch.long)

        return image, label_tensor, text

    def _resize(self, image):
        h, w = image.shape
        scale = self.target_height / h
        new_w = int(w * scale)
        image = cv2.resize(image, (new_w, self.target_height))

        # Pad or crop to target width
        if new_w < self.target_width:
            pad = self.target_width - new_w
            image = np.pad(image, ((0, 0), (0, pad)),
                           constant_values=255)
        else:
            image = image[:, :self.target_width]
        return image

    def _augment(self, image):
        """Light augmentation for historical docs — no flipping."""
        import random
        # Random brightness
        delta = random.randint(-20, 20)
        image = np.clip(image.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        # Random Gaussian noise
        if random.random() < 0.3:
            noise = np.random.normal(0, 5, image.shape).astype(np.int32)
            image = np.clip(image.astype(np.int32) + noise, 0, 255).astype(np.uint8)
        return image


def collate_fn(batch):
    """Custom collate to handle variable-length labels for CTC."""
    images, labels, texts = zip(*batch)
    images = torch.stack(images, 0)
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels_concat = torch.cat(labels)
    return images, labels_concat, label_lengths, texts


print('✓ dataset.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/dataset.py


In [48]:
%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/weighted_ctc.py
"""
src/weighted_ctc.py
Weighted CTC Loss for rare character recognition.

Strategy: Instead of modifying log_probs (which inflates loss values),
we apply per-sample reweighting AFTER computing standard CTC loss.
This keeps loss in normal range (5-30) while still boosting rare chars.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import math


def compute_char_weights(label_file: str, char_to_idx: dict,
                         smoothing: float = 1.0,
                         max_weight: float = 5.0) -> torch.Tensor:
    """
    Compute per-character inverse-frequency weights from training labels.
    """
    counts = Counter()
    total = 0

    with open(label_file, encoding="utf-8") as f:
        for line in f:
            if "	" not in line:
                continue
            text = line.strip().split("	", 1)[1]
            for ch in text:
                if ch in char_to_idx:
                    counts[char_to_idx[ch]] += 1
                    total += 1

    vocab_size = len(char_to_idx) + 1
    weights = torch.ones(vocab_size)
    weights[0] = 1.0  # blank always 1

    for idx in range(1, vocab_size):
        count = counts.get(idx, 0) + smoothing
        freq = count / (total + smoothing * vocab_size)
        weight = 1.0 / (freq * vocab_size)
        weights[idx] = min(weight, max_weight)

    # Normalize: mean weight = 1 so overall loss scale is preserved
    weights[1:] = weights[1:] / weights[1:].mean()

    print(f"  Char weights computed over {total} characters")
    print(f"  Weight range: [{weights[1:].min():.3f}, {weights[1:].max():.3f}]")
    return weights


class WeightedCTCLoss(nn.Module):
    """
    Weighted CTC: applies per-target-character weighting to the loss.
    Each sample loss is scaled by the mean weight of its target characters.
    This keeps loss values in the normal CTC range while boosting rare chars.
    """

    def __init__(self, char_weights: torch.Tensor):
        super().__init__()
        self.register_buffer("char_weights", char_weights)
        self.ctc = nn.CTCLoss(blank=0, zero_infinity=True, reduction="none")

    def forward(self, log_probs, targets, input_lengths, target_lengths):
        # Standard CTC loss per sample (on CPU as required by PyTorch)
        device = log_probs.device
        targets = targets.to(device)
        input_lengths = input_lengths.to(device)
        target_lengths = target_lengths.to(device)
        per_sample_loss = self.ctc(
            log_probs,
            targets,
            input_lengths,
            target_lengths
        )  # shape: (B,)

        # Compute per-sample weight = mean weight of its target characters

        weights = self.char_weights
        B = per_sample_loss.shape[0]
        sample_weights = torch.ones(B, device=device)



        offset = 0
        for b in range(B):
            length = target_lengths[b].item()
            if length > 0:
                chars = targets[offset: offset + length].to(device)
                char_w = weights[chars.clamp(0, len(weights)-1)]
                sample_weights[b] = char_w.mean()
            offset += length

        # Weighted mean loss — stays in normal CTC range
        weighted_loss = (per_sample_loss * sample_weights).mean()
        return weighted_loss


print("✓ weighted_ctc.py saved")


Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/weighted_ctc.py


In [49]:
%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/beam_decoder.py
"""
src/beam_decoder.py
Constrained Beam Search Decoder with Renaissance Spanish Lexicon.

Why: Greedy CTC decoding picks the single most likely char at each step,
     which ignores word-level context and produces hallucinated sequences.
     Beam search explores multiple hypotheses simultaneously and uses a
     lexicon to constrain outputs to real Spanish words.

Strategy:
  1. Run standard CTC beam search (width=10) on CRNN log-probs
  2. At word boundaries, score each hypothesis against the Spanish lexicon
  3. Heavily penalize hypotheses containing out-of-vocabulary words
  4. Return the highest-scoring valid sequence
"""

import torch
import math
from typing import List, Set, Dict, Tuple
from collections import defaultdict


class BeamDecoder:
    """
    CTC Beam Search Decoder with lexicon constraint.

    Args:
        idx_to_char:    mapping from index to character
        vocabulary:     set of valid Spanish words (lexicon)
        beam_width:     number of beams to maintain
        lm_weight:      weight for lexicon scoring (0 = pure CTC)
        blank_idx:      CTC blank token index
    """

    def __init__(self, idx_to_char: dict, vocabulary: Set[str],
                 beam_width: int = 10, lm_weight: float = 0.5,
                 blank_idx: int = 0):
        self.idx_to_char = idx_to_char
        self.vocabulary = {w.lower() for w in vocabulary}
        self.beam_width = beam_width
        self.lm_weight = lm_weight
        self.blank_idx = blank_idx

    def decode(self, log_probs: torch.Tensor) -> str:
        """
        Decode a single sequence.

        Args:
            log_probs: (T, vocab_size) log-softmax probabilities

        Returns:
            decoded string
        """
        T, vocab_size = log_probs.shape
        probs = log_probs.exp().cpu().numpy()

        # Beam: dict of {sequence: (score, last_char)}
        # sequence is tuple of char indices (no blanks, no repeats yet)
        beams = {(): (0.0, self.blank_idx)}

        for t in range(T):
            new_beams = defaultdict(lambda: -float("inf"))

            for seq, (score, last_char) in beams.items():
                for c in range(vocab_size):
                    p = float(probs[t, c])
                    if p < 1e-10:
                        continue
                    log_p = math.log(p)

                    if c == self.blank_idx:
                        # Blank: keep sequence, reset last_char
                        key = (seq, self.blank_idx)
                        candidate = score + log_p
                        if candidate > new_beams[seq]:
                            new_beams[seq] = candidate
                    elif c == last_char:
                        # Repeat without blank: skip (CTC rule)
                        continue
                    else:
                        new_seq = seq + (c,)
                        candidate = score + log_p
                        if candidate > new_beams[new_seq]:
                            new_beams[new_seq] = candidate

            # Keep top beam_width beams
            sorted_beams = sorted(new_beams.items(),
                                  key=lambda x: x[1], reverse=True)
            beams = {}
            for seq, score in sorted_beams[:self.beam_width]:
                last = seq[-1] if seq else self.blank_idx
                beams[seq] = (score, last)

        # Re-score with lexicon constraint
        best_seq, best_score = self._rescore(beams)
        return self._indices_to_text(best_seq)

    def _rescore(self, beams: dict) -> Tuple:
        """Apply lexicon penalty to final beams."""
        best_seq = ()
        best_score = -float("inf")

        for seq, (score, _) in beams.items():
            text = self._indices_to_text(seq)
            lex_score = self._lexicon_score(text)
            total = score + self.lm_weight * lex_score
            if total > best_score:
                best_score = total
                best_seq = seq

        return best_seq, best_score

    def _lexicon_score(self, text: str) -> float:
        """
        Score based on fraction of words in vocabulary.
        Returns value in [-1, 0]: 0 = all words valid, -1 = none valid.
        """
        words = text.lower().split()
        if not words:
            return 0.0
        in_vocab = sum(1 for w in words if w in self.vocabulary)
        return (in_vocab / len(words)) - 1.0

    def _indices_to_text(self, indices: tuple) -> str:
        chars = []
        for idx in indices:
            ch = self.idx_to_char.get(idx, "")
            chars.append(ch)
        return "".join(chars)

    def decode_batch(self, log_probs: torch.Tensor) -> List[str]:
        """
        Decode a batch.
        log_probs: (T, B, vocab_size)
        """
        T, B, V = log_probs.shape
        results = []
        for b in range(B):
            results.append(self.decode(log_probs[:, b, :]))
        return results


def build_spanish_lexicon(labels_file: str,
                           extra_words: List[str] = None) -> Set[str]:
    """
    Build a Renaissance Spanish lexicon from training transcriptions.
    Optionally augment with a provided word list.
    """
    import re
    vocab = set()

    with open(labels_file, encoding="utf-8") as f:
        for line in f:
            if "	" not in line:
                continue
            text = line.strip().split("	", 1)[1]
            words = re.findall(r"[\wÀ-ɏ]+", text.lower())
            vocab.update(words)

    if extra_words:
        vocab.update(w.lower() for w in extra_words)

    print(f"✓ Spanish lexicon built: {len(vocab)} unique words")
    return vocab


print("✓ beam_decoder.py saved")


Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/beam_decoder.py


In [50]:
%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/trainer.py
"""
src/trainer.py
Training loop for CRNN with Weighted CTC loss.
"""

import sys
sys.path.insert(0, '/content/drive/MyDrive/OCR_Pipeline_Research')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import editdistance
import matplotlib.pyplot as plt
from pathlib import Path
import logging

from src.crnn_model import CRNN
from src.dataset import OCRDataset, collate_fn
from src.charset import decode, VOCAB_SIZE, char_to_idx
from src.weighted_ctc import WeightedCTCLoss, compute_char_weights
from src.beam_decoder import BeamDecoder, build_spanish_lexicon

logger = logging.getLogger(__name__)


class CRNNTrainer:

    def __init__(self, labels_file, image_dir, save_dir,
                 hidden_size=256, num_rnn_layers=2,
                 batch_size=16, lr=1e-3, device=None,
                 use_weighted_loss=True, use_beam_decode=True):

        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.use_beam_decode = use_beam_decode

        self.device = device or (
            'cuda' if torch.cuda.is_available() else 'cpu'
        )
        print(f'Using device: {self.device}')

        # Dataset
        full_dataset = OCRDataset(labels_file, image_dir, augment=True)
        val_size = max(1, int(0.2 * len(full_dataset)))
        train_size = len(full_dataset) - val_size
        train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
        val_ds.dataset.augment = False

        self.train_loader = DataLoader(
            train_ds, batch_size=batch_size,
            shuffle=True, collate_fn=collate_fn, num_workers=0
        )
        self.val_loader = DataLoader(
            val_ds, batch_size=batch_size,
            shuffle=False, collate_fn=collate_fn, num_workers=0
        )
        print(f'Train: {train_size} | Val: {val_size} samples')

        # Model
        self.model = CRNN(
            vocab_size=VOCAB_SIZE,
            hidden_size=hidden_size,
            num_rnn_layers=num_rnn_layers
        ).to(self.device)

        # ── Weighted CTC Loss ──────────────────────────────────────────
        if use_weighted_loss:
            print('Computing character weights for rare letterforms...')
            char_weights = compute_char_weights(labels_file, char_to_idx)
            self.criterion = WeightedCTCLoss(char_weights).to(self.device)
            print('✓ Using WeightedCTCLoss (rare chars boosted)')
        else:
            self.criterion = nn.CTCLoss(blank=0, zero_infinity=True)
            print('✓ Using standard CTCLoss')

        # ── Spanish Lexicon for Beam Decoder ──────────────────────────
        if use_beam_decode:
            print('Building Renaissance Spanish lexicon...')
            self.vocabulary = build_spanish_lexicon(labels_file)
            from src.charset import idx_to_char
            self.beam_decoder = BeamDecoder(
                idx_to_char=idx_to_char,
                vocabulary=self.vocabulary,
                beam_width=10,
                lm_weight=0.5
            )
            print('✓ Beam decoder ready')
        else:
            self.beam_decoder = None

        # Optimizer + scheduler
        self.optimizer = optim.AdamW(
            self.model.parameters(), lr=lr, weight_decay=1e-4
        )
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, patience=5, factor=0.5
        )
        self.history = {'train_loss': [], 'val_cer_greedy': [], 'val_cer_beam': []}

    def train(self, num_epochs=50):
        best_cer = float('inf')

        for epoch in range(1, num_epochs + 1):
            train_loss = self._train_epoch()
            greedy_cer, beam_cer = self._validate()

            self.history['train_loss'].append(train_loss)
            self.history['val_cer_greedy'].append(greedy_cer)
            self.history['val_cer_beam'].append(beam_cer)
            self.scheduler.step(beam_cer)

            print(f'Epoch [{epoch:3d}/{num_epochs}] '
                  f'Loss: {train_loss:.4f} | '
                  f'Greedy CER: {greedy_cer:.4f} | '
                  f'Beam CER: {beam_cer:.4f}')

            # Save best on beam CER
            if beam_cer < best_cer:
                best_cer = beam_cer
                torch.save(self.model.state_dict(),
                           self.save_dir / 'best_crnn.pth')
                print(f'  ✓ Saved best model (Beam CER: {best_cer:.4f})')

        self._plot_training()
        return best_cer

    def _train_epoch(self):
        print("🚀 Entered training loop")

        self.model.train()
        total_loss = 0
        print("➡️ Starting batch loop")
        for images, labels, label_lengths, _ in self.train_loader:
            images = images.to(self.device)
            labels = labels.to(self.device)

            self.optimizer.zero_grad()
            log_probs = self.model(images)
            T, B, _ = log_probs.shape
            input_lengths = torch.full((B,), T, dtype=torch.long).to(self.device)

            loss = self.criterion(log_probs, labels, input_lengths, label_lengths)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 5.0)
            self.optimizer.step()
            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    def _validate(self):
        self.model.eval()
        greedy_cer_total = 0
        beam_cer_total = 0
        count = 0

        with torch.no_grad():
            for images, _, _, texts in self.val_loader:
                images = images.to(self.device)
                log_probs = self.model(images)   # (T, B, vocab)

                # Greedy decode
                preds_greedy = log_probs.argmax(2).permute(1, 0)
                for pred, gt in zip(preds_greedy, texts):
                    pred_text = decode(pred.cpu().tolist())
                    greedy_cer_total += editdistance.eval(pred_text, gt) / max(len(gt), 1)

                # Beam decode (with lexicon constraint)
                if self.beam_decoder:
                    beam_preds = self.beam_decoder.decode_batch(log_probs)
                    for pred_text, gt in zip(beam_preds, texts):
                        beam_cer_total += editdistance.eval(pred_text, gt) / max(len(gt), 1)
                else:
                    beam_cer_total = greedy_cer_total

                count += len(texts)

        n = max(count, 1)
        return greedy_cer_total / n, beam_cer_total / n

    def _plot_training(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(self.history['train_loss'], label='Train Loss')
        ax1.set_title('Training Loss (Weighted CTC)')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
        ax1.legend(); ax1.grid(True)

        ax2.plot(self.history['val_cer_greedy'], label='Greedy CER', linestyle='--')
        ax2.plot(self.history['val_cer_beam'],   label='Beam CER (lexicon)', linewidth=2)
        ax2.set_title('Validation CER: Greedy vs Beam Search')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('CER')
        ax2.legend(); ax2.grid(True)

        plt.tight_layout()
        plt.savefig(self.save_dir / 'training_curves.png', dpi=150)
        plt.show()
        print('✓ Training curves saved')


print('✓ trainer.py saved')


Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/trainer.py


In [51]:
print("➡️ Starting batch loop")

➡️ Starting batch loop


In [52]:
# ============================================================
# CELL 10 — METRICS
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/metrics.py
"""
src/metrics.py
OCR Evaluation Metrics.

CER (Character Error Rate):
    CER = edit_distance(predicted, ground_truth) / len(ground_truth)
    Range: 0.0 (perfect) to 1.0+ (completely wrong)

WER (Word Error Rate):
    WER = edit_distance(pred_words, gt_words) / len(gt_words)

Both based on Levenshtein edit distance (insertions, deletions, substitutions).
"""

import editdistance


class Metrics:

    @staticmethod
    def cer(predicted: str, ground_truth: str) -> float:
        if not ground_truth:
            return 0.0 if not predicted else 1.0
        return editdistance.eval(predicted, ground_truth) / len(ground_truth)

    @staticmethod
    def wer(predicted: str, ground_truth: str) -> float:
        pred_words = predicted.split()
        gt_words = ground_truth.split()
        if not gt_words:
            return 0.0 if not pred_words else 1.0
        return editdistance.eval(pred_words, gt_words) / len(gt_words)

    @staticmethod
    def accuracy(predicted: str, ground_truth: str) -> float:
        """Character-level accuracy (complement of CER, capped at 0)."""
        return max(0.0, 1.0 - Metrics.cer(predicted, ground_truth))


print('✓ metrics.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/metrics.py


In [53]:
# ============================================================
# CELL 11 — LLM POST-CORRECTION (Gemini / GPT-4o-mini)
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/llm_corrector.py
"""
src/llm_corrector.py
LLM post-correction stage for OCR output.
Supports: Gemini (recommended, free tier) or OpenAI GPT-4o-mini.

Role in pipeline: LATE-STAGE cleanup only.
The CRNN does the heavy lifting; LLM corrects residual errors
using knowledge of 17th century Spanish language patterns.
"""

import os


class LLMCorrector:

    SYSTEM_PROMPT = (
        'You are an expert transcriber of 17th century Spanish printed sources. '
        'You will receive OCR output that may contain character recognition errors. '
        'Fix only clear OCR mistakes (e.g. rn->m, 0->o, I->l) using your knowledge '
        'of early modern Spanish spelling and vocabulary. '
        'Do NOT modernize spelling. Do NOT change meaning. '
        'Return only the corrected text, nothing else.'
    )

    def __init__(self, provider='gemini'):
        self.provider = provider
        self._client = None

        if provider == 'gemini':
            self._init_gemini()
        elif provider == 'openai':
            self._init_openai()
        else:
            raise ValueError(f'Unknown provider: {provider}')

    def _init_gemini(self):
        try:
            import google.generativeai as genai
            api_key = os.environ.get('GEMINI_API_KEY', '')
            genai.configure(api_key=api_key)
            self._client = genai.GenerativeModel('gemini-2.0-flash')
            print('✓ Gemini client initialized')
        except Exception as e:
            print(f'⚠ Gemini init failed: {e}')
            self._client = None

    def _init_openai(self):
        try:
            from openai import OpenAI
            self._client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', ''))
            print('✓ OpenAI client initialized')
        except Exception as e:
            print(f'⚠ OpenAI init failed: {e}')
            self._client = None

    def correct(self, text: str) -> str:
        if not text or not text.strip() or self._client is None:
            return text
        try:
            if self.provider == 'gemini':
                return self._correct_gemini(text)
            else:
                return self._correct_openai(text)
        except Exception as e:
            print(f'LLM correction failed: {e}')
            return text

    def _correct_gemini(self, text: str) -> str:
        prompt = f'{self.SYSTEM_PROMPT}\n\nOCR text to correct:\n{text}'
        response = self._client.generate_content(prompt)
        return response.text.strip()

    def _correct_openai(self, text: str) -> str:
        response = self._client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': self.SYSTEM_PROMPT},
                {'role': 'user', 'content': text}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content.strip()


print('✓ llm_corrector.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/llm_corrector.py


In [54]:
# ============================================================
# CELL 12 — API KEY SETUP (Gemini recommended — free tier)
# ============================================================

import os

# Install Gemini SDK
!pip install -q google-generativeai

try:
    from google.colab import userdata
    # Add GEMINI_API_KEY to Colab Secrets (left sidebar > key icon)
    # Get free key at: https://aistudio.google.com/app/apikey
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('✓ Gemini API key loaded')
except Exception as e:
    print(f'⚠ Set GEMINI_API_KEY in Colab Secrets: {e}')

✓ Gemini API key loaded


In [55]:
# ============================================================
# CELL 13 — FULL PIPELINE (inference + evaluation)
# ============================================================

%%writefile /content/drive/MyDrive/OCR_Pipeline_Research/src/pipeline.py
"""
src/pipeline.py
Full OCR pipeline: Preprocess -> Region Detect -> Line Segment -> CRNN -> LLM
"""

import sys
sys.path.insert(0, '/content/drive/MyDrive/OCR_Pipeline_Research')

import cv2
import torch
import numpy as np
import logging
from pathlib import Path
from typing import List, Dict

from src.preprocess import ImagePreprocessor
from src.region_detector import TextRegionDetector
from src.line_segmenter import LineSegmenter
from src.crnn_model import CRNN
from src.charset import decode, VOCAB_SIZE
from src.llm_corrector import LLMCorrector

logger = logging.getLogger(__name__)


class OCRPipeline:

    def __init__(self, model_path: str, use_llm=True,
                 llm_provider='gemini', device=None):

        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        self.preprocessor = ImagePreprocessor(target_height=64)
        self.region_detector = TextRegionDetector(margin_ratio=0.15)
        self.line_segmenter = LineSegmenter(min_line_height=15, gap_threshold=5)

        # Load trained CRNN
        self.model = CRNN(vocab_size=VOCAB_SIZE).to(self.device)
        if Path(model_path).exists():
            self.model.load_state_dict(
                torch.load(model_path, map_location=self.device)
            )
            print(f'✓ CRNN model loaded from {model_path}')
        else:
            print(f'⚠ Model not found at {model_path} — using untrained model')
        self.model.eval()

        self.llm = LLMCorrector(provider=llm_provider) if use_llm else None

    def process_page(self, image_path: str, visualize=False) -> Dict:
        """
        Process a full page image through the complete pipeline.
        Returns dict with per-stage text output.
        """
        # Stage 1: Preprocess
        preprocessed, _ = self.preprocessor.process(image_path)

        # Stage 2: Detect main text region (ignore marginalia)
        text_region, bbox = self.region_detector.detect(
            preprocessed, visualize=visualize
        )

        # Stage 3: Segment into lines
        line_images, line_coords = self.line_segmenter.segment(
            text_region, visualize=visualize
        )

        if not line_images:
            return {'raw': '', 'llm': '', 'lines': []}

        # Stage 4: CRNN inference on each line
        raw_lines = []
        for line_img in line_images:
            text = self._recognize_line(line_img)
            raw_lines.append(text)

        raw_text = '\n'.join(raw_lines)

        # Stage 5: LLM post-correction
        llm_text = raw_text
        if self.llm:
            llm_text = self.llm.correct(raw_text)

        return {
            'raw': raw_text,
            'llm': llm_text,
            'lines': raw_lines,
            'num_lines': len(line_images),
            'text_bbox': bbox,
        }

    def _recognize_line(self, line_img: np.ndarray) -> str:
        """Run CRNN inference on a single text line image."""
        # Resize to CRNN input height
        line_img = self.preprocessor.resize_for_crnn(line_img)

        # Normalize + tensorize
        img_tensor = torch.tensor(
            line_img.astype(np.float32) / 255.0
        ).unsqueeze(0).unsqueeze(0).to(self.device)   # (1, 1, H, W)

        with torch.no_grad():
            log_probs = self.model(img_tensor)   # (T, 1, vocab)
            pred = log_probs.argmax(2).squeeze(1)  # (T,)

        return decode(pred.cpu().tolist())


print('✓ pipeline.py saved')

Overwriting /content/drive/MyDrive/OCR_Pipeline_Research/src/pipeline.py


In [ ]:
!cd /content/drive/MyDrive/OCR_Pipeline_Research
!zip -r data.zip data

In [ ]:
!unzip /content/drive/MyDrive/OCR_Pipeline_Research/data.zip -d /content/

In [ ]:
!zip -r src.zip src/

In [ ]:
# ============================================================
# CELL 15 — TRAIN THE CRNN
# ============================================================

import sys, os
BASE_PATH = '/content/drive/MyDrive/OCR_Pipeline_Research'
sys.path.insert(0, BASE_PATH)
os.chdir(BASE_PATH)
from pathlib import Path
Path(BASE_PATH, 'src', '__init__.py').touch(exist_ok=True)




from src.trainer import CRNNTrainer

trainer = CRNNTrainer(
    labels_file="/content/drive/MyDrive/OCR_Pipeline_Research/data/annotations/line_labels.txt",
    image_dir="/content/drive/MyDrive/OCR_Pipeline_Research/data/rodrigo/extracted/images",
    save_dir=f'{BASE_PATH}/models/crnn',
    hidden_size=256,
    num_rnn_layers=2,
    batch_size=32,
    lr=1e-4,
    use_beam_decode=True
)

best_cer = trainer.train(num_epochs=100)
print(f'\n✓ Training complete. Best CER: {best_cer:.4f}')

In [ ]:
# ============================================================
# CELL 16 — FULL PIPELINE EVALUATION
# ============================================================

import sys, os, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE_PATH = '/content/drive/MyDrive/OCR_Pipeline_Research'
sys.path.insert(0, BASE_PATH)
os.chdir(BASE_PATH)
Path(BASE_PATH, 'src', '__init__.py').touch(exist_ok=True)

from src.pipeline import OCRPipeline
from src.metrics import Metrics

# Load ground truth
gt = {}
with open(f'{BASE_PATH}/data/ground_truth_full.txt', encoding='utf-8') as f:
    for line in f:
        if line.strip() and not line.startswith('#'):
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                gt[parts[0].strip()] = parts[1].strip()

# Init pipeline
pipeline = OCRPipeline(
    model_path=f'{BASE_PATH}/models/crnn/best_crnn.pth',
    use_llm=True,
    llm_provider='gemini'
)

# Run evaluation
results = []
image_dir = Path(f'{BASE_PATH}/data/images')
images = list(image_dir.rglob('*.png')) + list(image_dir.rglob('*.jpg'))

print(f'Evaluating {len(images)} images...\n')

for img_path in sorted(images):
    rel = str(img_path).replace(BASE_PATH + '/', '')
    if rel not in gt:
        print(f'  ⚠ No GT for {img_path.name}')
        continue

    try:
        result = pipeline.process_page(str(img_path), visualize=True)
    except Exception as e:
        print(f'  ❌ {img_path.name}: {e}')
        continue

    gt_text = gt[rel]

    for stage in ['raw', 'llm']:
        text = result[stage]
        cer = Metrics.cer(text, gt_text)
        wer = Metrics.wer(text, gt_text)
        acc = Metrics.accuracy(text, gt_text)
        results.append({
            'filename': img_path.name,
            'stage': stage,
            'cer': cer,
            'wer': wer,
            'accuracy': acc
        })

    print(f'{img_path.name}')
    print(f'  RAW → CER: {Metrics.cer(result["raw"], gt_text):.3f}')
    print(f'  LLM → CER: {Metrics.cer(result["llm"], gt_text):.3f}')
    print()

# Summary table
df = pd.DataFrame(results)
print('='*60)
print('RESULTS BY STAGE')
print('='*60)
print(df.groupby('stage')[['cer', 'wer', 'accuracy']].mean().round(4))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('stage')['cer'].mean().plot(kind='bar', ax=axes[0],
    color=['steelblue', 'coral'], title='Mean CER by Stage (lower=better)')
axes[0].set_ylabel('CER')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('stage')['accuracy'].mean().plot(kind='bar', ax=axes[1],
    color=['steelblue', 'coral'], title='Mean Accuracy by Stage (higher=better)')
axes[1].set_ylabel('Accuracy')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/results/evaluation_results.png', dpi=150)
plt.show()

print(f'\n✓ Results saved')

## Discussion

### Architecture Choice: CRNN with CTC
A **CNN-RNN (CRNN)** architecture was chosen for this task because it is specifically designed for sequence recognition from images without requiring character-level segmentation:
- The **VGG-style CNN** extracts local visual features column by column, capturing stroke patterns, serifs, and letterforms typical of 17th century Spanish print
- The **Bidirectional LSTM** reads the feature sequence in both directions, capturing ligatures and context-dependent letterform variations common in early modern typography
- **CTC loss** allows end-to-end training using only line-level text labels — no bounding box annotations needed per character

### LLM Post-Correction Role
Gemini/GPT-4o-mini is used as a **late-stage** correction step only (as required by Test I). Its role is to:
- Resolve common OCR confusions (rn→m, 0→o, I→l)
- Restore period-appropriate Spanish spelling using language model knowledge
- It does NOT replace the CRNN — it refines its output

### Evaluation Metrics
- **CER (Character Error Rate):** Primary metric. Measures edit distance at character level normalized by GT length. Target < 0.10 (90% accuracy)
- **WER (Word Error Rate):** Secondary metric. More sensitive to spacing errors and word boundary recognition
- Both computed per stage (raw CRNN output vs LLM-corrected) to quantify LLM contribution

### Text Region Detection
A morphology + contour-based approach detects the main text block and crops out marginalia, headers, and decorative elements, ensuring the model only processes the main body text as specified.